In [ ]:
"""Day 26 / Session 3 - Lab Assignment 2: Decoding Knobs from Scratch - Temperature, Top-k & Top-p (Nucleus) Filtering."""
import numpy as np


def softmax(logits):
    z = np.asarray(logits, dtype=float)
    e = np.exp(z - np.max(z))                                # stable: subtract the max first
    return e / e.sum()


def apply_temperature(logits, T):
    return softmax(np.asarray(logits, dtype=float) / T)      # T<1 sharpens, T>1 flattens


def _renormalize(kept):
    return kept / kept.sum()


def top_k_filter(probs, k):
    probs = np.asarray(probs, dtype=float)
    k = max(1, int(k))
    order = np.argsort(-probs, kind="stable")                # highest probability first, ties by original index
    kept = np.zeros_like(probs)
    keep_idx = order[:k]
    kept[keep_idx] = probs[keep_idx]
    return _renormalize(kept)


def nucleus_support(probs, p):
    probs = np.asarray(probs, dtype=float)
    order = np.argsort(-probs, kind="stable")
    cumulative = np.cumsum(probs[order])
    size = int(np.searchsorted(cumulative, p - 1e-12, side="left")) + 1     # smallest prefix with cumulative >= p
    size = min(max(size, 1), len(probs))                                    # always at least the top token
    return np.sort(order[:size])


def top_p_filter(probs, p):
    probs = np.asarray(probs, dtype=float)
    keep_idx = nucleus_support(probs, p)
    kept = np.zeros_like(probs)
    kept[keep_idx] = probs[keep_idx]
    return _renormalize(kept)
